In [4]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import glob
from tqdm import tqdm
import albumentations as A
import random
import tensorflow as tf

In [5]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Use GPU 0 (change if you have multiple GPUs)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU detected: {gpus[0].name}")
    try:
        # Enable memory growth to prevent OOM errors
        tf.config.experimental.set_memory_growth(gpus[0], True)
        print("✅ Memory growth enabled for GPU")
    except Exception as e:
        print(f"⚠️ Could not set memory growth: {e}")
else:
    print("⚠️ No GPU detected, using CPU")

✅ GPU detected: /physical_device:GPU:0
✅ Memory growth enabled for GPU


In [6]:
# Define main folder
base_path = r"D:\PHD Project"

# Define class names (folder names)
classes = ['major-damage', 'minor-damage', 'unknown']

# Initialize lists
images = []
labels = []

# Load images from each folder
for label, cls in enumerate(classes):
    folder_path = os.path.join(base_path, cls, "images")
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_path = os.path.join(folder_path, filename)
            img = cv2.imread(img_path)
            img = cv2.resize(img, (128, 128))
            images.append(img)
            labels.append(label)

# Convert to NumPy arrays
images = np.array(images)
labels = np.array(labels)

print("Total images:", len(images))
print("Labels:", np.unique(labels, return_counts=True))

Total images: 611
Labels: (array([0, 1, 2]), array([ 92, 132, 387], dtype=int64))


In [7]:
import albumentations as A
import cv2
import os
from tqdm import tqdm

# Augmentation pipeline (no tensors, pure NumPy)
augmentation = A.Compose([
    A.Resize(128, 128),
    A.Rotate(limit=15),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Affine(shear=10, p=0.5)
])

# Base and output directories
base_dir = r"D:\PHD Project"
output_dir = os.path.join(base_dir, "Balanced")
os.makedirs(output_dir, exist_ok=True)

# Class names and counts
class_counts = {
    "major-damage": 92,
    "minor-damage": 132,
    "unknown": 387
}
target_count = 387

for cls, count in class_counts.items():
    print(f"\nProcessing class '{cls}' ...")

    img_dir = os.path.join(base_dir, cls, "images")
    mask_dir = os.path.join(base_dir, cls, "masks")

    out_img_dir = os.path.join(output_dir, cls, "images")
    out_mask_dir = os.path.join(output_dir, cls, "masks")
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_mask_dir, exist_ok=True)

    # Get all image names
    images = sorted(os.listdir(img_dir))
    masks = sorted(os.listdir(mask_dir))

    # Copy originals
    for img_name, mask_name in zip(images, masks):
        image = cv2.imread(os.path.join(img_dir, img_name))
        mask = cv2.imread(os.path.join(mask_dir, mask_name), cv2.IMREAD_GRAYSCALE)
        cv2.imwrite(os.path.join(out_img_dir, img_name), image)
        cv2.imwrite(os.path.join(out_mask_dir, mask_name), mask)

    # Augment only if needed
    if count < target_count:
        needed = target_count - count
        idx = 0
        for i in tqdm(range(needed), desc=f"Augmenting {cls}"):
            img_name = images[idx % len(images)]
            mask_name = masks[idx % len(masks)]
            image = cv2.imread(os.path.join(img_dir, img_name))
            mask = cv2.imread(os.path.join(mask_dir, mask_name), cv2.IMREAD_GRAYSCALE)

            augmented = augmentation(image=image, mask=mask)
            aug_img = augmented["image"]
            aug_mask = augmented["mask"]

            new_name = img_name.replace(".png", f"_aug{i}.png")
            cv2.imwrite(os.path.join(out_img_dir, new_name), aug_img)
            cv2.imwrite(os.path.join(out_mask_dir, new_name), aug_mask)

            idx += 1

print("\n✅ All classes balanced successfully! Saved in 'Balanced' folder.")


Processing class 'major-damage' ...


Augmenting major-damage: 100%|██████████| 295/295 [00:00<00:00, 461.04it/s]



Processing class 'minor-damage' ...


Augmenting minor-damage: 100%|██████████| 255/255 [00:00<00:00, 480.06it/s]



Processing class 'unknown' ...

✅ All classes balanced successfully! Saved in 'Balanced' folder.


In [8]:
import os
import shutil
import random
from tqdm import tqdm

# Base directory (your balanced dataset)
base_dir = r"D:\PHD Project\Balanced"
output_dir = r"D:\PHD Project\Balanced_Split"

# Split ratio (80% train, 20% test)
train_ratio = 0.8

# Create output folders
splits = ["train", "test"]
for split in splits:
    for cls in ["major-damage", "minor-damage", "unknown"]:
        os.makedirs(os.path.join(output_dir, split, cls), exist_ok=True)

# Loop over classes
for cls in ["major-damage", "minor-damage", "unknown"]:
    img_dir = os.path.join(base_dir, cls, "images")
    images = sorted(os.listdir(img_dir))
    random.shuffle(images)

    total = len(images)
    train_end = int(train_ratio * total)

    train_imgs = images[:train_end]
    test_imgs = images[train_end:]

    # Copy images into split folders
    for img_name in tqdm(train_imgs, desc=f"{cls} - train"):
        shutil.copy(os.path.join(img_dir, img_name),
                    os.path.join(output_dir, "train", cls, img_name))

    for img_name in tqdm(test_imgs, desc=f"{cls} - test"):
        shutil.copy(os.path.join(img_dir, img_name),
                    os.path.join(output_dir, "test", cls, img_name))

print("\n✅ Dataset split into Train/Test successfully!")
print("📁 Saved in:", output_dir)

unknown - test: 100%|██████████| 78/78 [00:01<00:00, 47.83it/s]


✅ Dataset split into Train/Test successfully!
📁 Saved in: D:\PHD Project\Balanced_Split


In [9]:
import os
import cv2
import numpy as np
from tqdm import tqdm

base_path = r"D:\PHD Project\Balanced_Split"
classes = ['major-damage', 'minor-damage', 'unknown']

def load_images_from_split(split):
    images = []
    labels = []
    for label, cls in enumerate(classes):
        folder = os.path.join(base_path, split, cls)
        for filename in os.listdir(folder):
            if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(folder, filename)
                img = cv2.imread(img_path)
                img = cv2.resize(img, (128, 128))
                images.append(img)
                labels.append(label)
    return np.array(images), np.array(labels)

# Load train and test sets
X_train, y_train = load_images_from_split("train")
X_test, y_test = load_images_from_split("test")

print("✅ Train images:", len(X_train))
print("✅ Test images:", len(X_test))

✅ Train images: 1159
✅ Test images: 635


In [10]:
def extract_color_histogram(images, bins=(8, 8, 8)):
    features = []
    for img in tqdm(images, desc="Extracting features"):
        img = (img * 255).astype(np.uint8) if img.max() <= 1 else img
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        hist = cv2.calcHist([hsv], [0, 1, 2], None, bins, [0, 256, 0, 256, 0, 256])
        hist = cv2.normalize(hist, hist).flatten()
        features.append(hist)
    return np.array(features)

X_train_features = extract_color_histogram(X_train)
X_test_features = extract_color_histogram(X_test)

Extracting features: 100%|██████████| 635/635 [00:00<00:00, 5678.04it/s]


In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train_features, y_train)
y_pred = rf.predict(X_test_features)

print("✅ Accuracy:", accuracy_score(y_test, y_pred))
print("\n🧾 Classification Report:\n", classification_report(y_test, y_pred))
print("\n📉 Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

✅ Accuracy: 0.9637795275590552

🧾 Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.96      0.95       219
           1       0.99      0.93      0.96       203
           2       0.96      1.00      0.98       213

    accuracy                           0.96       635
   macro avg       0.97      0.96      0.96       635
weighted avg       0.96      0.96      0.96       635


📉 Confusion Matrix:
 [[210   1   8]
 [ 12 189   2]
 [  0   0 213]]


In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

In [16]:
# Define models to compare
models = {
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "SVM (RBF)": SVC(kernel='rbf', probability=True),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "XGBoost": XGBClassifier(eval_metric='mlogloss', use_label_encoder=False)
}

In [17]:
# Iterate through models
for name, model in models.items():
    print(f"\n🔹 Model: {name}")
    print("-" * 60)
    
    # Train model
    model.fit(X_train_features, y_train)
    
    # ---- Training evaluation ----
    y_train_pred = model.predict(X_train_features)
    train_acc = accuracy_score(y_train, y_train_pred)
    print(f"✅ Training Accuracy: {train_acc:.4f}")
    print("\n--- 🧾 Classification Report (Training) ---")
    print(classification_report(y_train, y_train_pred, target_names=['major-damage', 'minor-damage', 'unknown']))
    
    # ---- Testing evaluation ----
    y_test_pred = model.predict(X_test_features)
    test_acc = accuracy_score(y_test, y_test_pred)
    print(f"\n🧠 Testing Accuracy: {test_acc:.4f}")
    print("\n--- 📊 Classification Report (Testing) ---")
    print(classification_report(y_test, y_test_pred, target_names=['major-damage', 'minor-damage', 'unknown']))
    
    print("=" * 60)


🔹 Model: Random Forest
------------------------------------------------------------
✅ Training Accuracy: 1.0000

--- 🧾 Classification Report (Training) ---
              precision    recall  f1-score   support

major-damage       1.00      1.00      1.00       386
minor-damage       1.00      1.00      1.00       387
     unknown       1.00      1.00      1.00       386

    accuracy                           1.00      1159
   macro avg       1.00      1.00      1.00      1159
weighted avg       1.00      1.00      1.00      1159


🧠 Testing Accuracy: 0.9638

--- 📊 Classification Report (Testing) ---
              precision    recall  f1-score   support

major-damage       0.95      0.96      0.95       219
minor-damage       0.99      0.93      0.96       203
     unknown       0.96      1.00      0.98       213

    accuracy                           0.96       635
   macro avg       0.97      0.96      0.96       635
weighted avg       0.96      0.96      0.96       635


🔹 Model: 

d:\Anaconda\envs\tf_env\lib\site-packages\xgboost\training.py:199: UserWarning: [21:47:36] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


✅ Training Accuracy: 1.0000

--- 🧾 Classification Report (Training) ---
              precision    recall  f1-score   support

major-damage       1.00      1.00      1.00       386
minor-damage       1.00      1.00      1.00       387
     unknown       1.00      1.00      1.00       386

    accuracy                           1.00      1159
   macro avg       1.00      1.00      1.00      1159
weighted avg       1.00      1.00      1.00      1159


🧠 Testing Accuracy: 0.9827

--- 📊 Classification Report (Testing) ---
              precision    recall  f1-score   support

major-damage       1.00      0.95      0.98       219
minor-damage       0.98      1.00      0.99       203
     unknown       0.97      1.00      0.98       213

    accuracy                           0.98       635
   macro avg       0.98      0.98      0.98       635
weighted avg       0.98      0.98      0.98       635



In [20]:
import joblib

joblib.dump(models["Random Forest"], "random_forest.pkl")
joblib.dump(models["SVM (RBF)"], "svm_rbf.pkl")
joblib.dump(models["Logistic Regression"], "logistic_regression.pkl")
joblib.dump(models["XGBoost"], "xgboost.pkl")

['xgboost.pkl']